# 005 — Golden-set construction + human review (candidate taxonomy validation)

**The model proposes. The human decides.**

This notebook does NOT discover a new taxonomy and does NOT train a classifier.
It builds a **human-reviewable golden set of exactly 250 real conversations** that
stress-tests the **candidate taxonomy produced by notebook 004**, so a human can
confirm, reject, merge, split, or rename the proposed intents.

## What 005 consumes from 004 (no duplication of expensive work)

| Artifact from 004 | Location | Reused for |
|---|---|---|
| Full-conversation texts + candidate labels (`intent`, `domain`, `sub_intent`, `intent_confidence`, `interaction_type`) | `data/processed/conversations_with_intents_full_v1.jsonl` (26,388 conversations) | `candidate_intent` / `candidate_domain` / `candidate_sub_intent` suggestions; **PROVISIONAL** population distribution |
| Full-conversation FAISS store (768-d, `nomic-embed-text:latest`, preprocessing `full_conversation_roles_mentions_urls_v2`) | `data/faiss_full_conversation/` | Recover embedding vectors **directly from the index** — no re-embedding, no new vector store, no Ollama calls |
| Candidate taxonomy (9 intents; clusters 0+3 merged into *Positive experience*; k=10 KMeans) | 004 §"Final taxonomy and safe versioned output" | Displayed verbatim for KEEP / RENAME / MERGE / SPLIT / REMOVE review |
| Conversation schema (`conversation_id`, `messages[{tweet_id, author_id, role, inbound, created_at, text}]`) | `data/processed/conversations.jsonl` via 003 | Full chronological `CUSTOMER:` / `AGENT:` review rendering |

Geometry (cluster distances, ambiguity margins) is **recomputed deterministically**
(`KMeans k=10, n_init=20, random_state=42`) on the **recovered** vectors, reproducing
004's partition exactly. Candidate *labels* always come from 004's output file, never
from the recomputation.

## Sampling design (fixed)

| Group | n | Purpose |
|---|---|---|
| `representative` | 150 | Approximates the **PROVISIONAL** candidate population distribution (largest-remainder quotas) |
| `uncertainty` | 60 | Small distance-margin / boundary / multi-centroid cases (ambiguity, not calibrated probability) |
| `challenge` | 40 | Short, vague, multi-intent, sarcastic, outlier cases that deliberately stress the taxonomy |
| **Total** | **250** | One `golden_set.csv`; `sampling_group` preserved so downstream metrics can separate population-like (150) from enriched diagnostic (100) subsets |

> Terminology guardrails used throughout: `candidate_distribution` (unsupervised, provisional)
> vs `sampling_distribution` (our design) vs `human_annotated_distribution` (only after review).
> The 250-record mix is **never** called the true intent distribution.


In [1]:
import json
import pickle
import re
from collections import Counter, defaultdict
from pathlib import Path

import faiss
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

SEED = 42
rng = np.random.default_rng(SEED)

# ---- inputs (all produced by 003/004; nothing is re-embedded here) ----
CANDIDATE_PATH = Path("../data/processed/conversations_with_intents_full_v1.jsonl")
FAISS_DIR = Path("../data/faiss_full_conversation")
META_PATH = FAISS_DIR / "metadata.json"
MODEL = "nomic-embed-text:latest"
PREPROCESSING_VERSION = "full_conversation_roles_mentions_urls_v2"
SELECTED_K = 10  # must match 004's chosen k

# ---- golden-set design (fixed by spec) ----
GOLD_SET_SIZE = 250
N_REPRESENTATIVE = 150
N_UNCERTAINTY = 60
N_CHALLENGE = 40
assert N_REPRESENTATIVE + N_UNCERTAINTY + N_CHALLENGE == GOLD_SET_SIZE

GOLDEN_PATH = Path("../data/processed/golden_set.csv")

print(f"Design: {N_REPRESENTATIVE} representative + {N_UNCERTAINTY} uncertainty + {N_CHALLENGE} challenge = {GOLD_SET_SIZE}")
print(f"Output: {GOLDEN_PATH}")


Design: 150 representative + 60 uncertainty + 40 challenge = 250
Output: ..\data\processed\golden_set.csv


## Step 1 — Load 004 artifacts (reuse only; no re-embedding, no new index)

We load candidate labels from 004's versioned JSONL and recover the embedding matrix
from the existing FAISS index via `read_index` + `reconstruct` (no Ollama, no
`langchain` embedding calls). Integrity checks: model name, preprocessing version,
vector count, and exact conversation-id ordering.


In [2]:
def load_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def normalize_message_text(text):
    # identical to 004's normalization (URLs / mentions) so geometry matches
    text = str(text or "")
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " [MENTION] ", text)
    return re.sub(r"\s+", " ", text).strip()

def prepare_conversation_for_embedding(conversation):
    # identical to 004's embedding unit: full chronological conversation w/ role markers
    lines = []
    for m in conversation.get("messages", []):
        role = str(m.get("role", "unknown")).upper()
        lines.append(f"{role}: {normalize_message_text(m.get('text', ''))}")
    return "\n".join(lines)

def format_conversation_readable(record):
    # human-readable full conversation for review (RAW text, not normalized)
    blocks = []
    for m in record.get("messages", []):
        role = str(m.get("role", "unknown")).upper()
        blocks.append(f"{role}:\n{str(m.get('text', '')).strip()}")
    return "\n\n".join(blocks)

records = load_jsonl(CANDIDATE_PATH)
rec_by_id = {str(r["conversation_id"]): r for r in records}
print(f"Candidate records from 004: {len(records):,}")

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
assert meta["model"] == MODEL, meta["model"]
assert meta["preprocessing_version"] == PREPROCESSING_VERSION, meta["preprocessing_version"]
assert meta["representation_type"] == "full_conversation"
assert meta["embedded_count"] == len(records) == meta["conversation_count"]
file_ids = [str(r["conversation_id"]) for r in records]
assert file_ids == [str(i) for i in meta["conversation_ids"]], "FAISS order != candidate file order"
print("FAISS metadata checks passed:", {k: meta[k] for k in
      ["model", "dimension", "preprocessing_version", "representation_type", "conversation_count"]})

index = faiss.read_index(str(FAISS_DIR / "index.faiss"))
assert index.ntotal == len(records), (index.ntotal, len(records))
X = np.zeros((index.ntotal, index.d), dtype=np.float32)
for i in range(index.ntotal):
    X[i] = index.reconstruct(i)
assert np.isfinite(X).all()
print(f"Recovered embedding matrix from FAISS (no re-embedding): {X.shape}")


Candidate records from 004: 26,388
FAISS metadata checks passed: {'model': 'nomic-embed-text:latest', 'dimension': 768, 'preprocessing_version': 'full_conversation_roles_mentions_urls_v2', 'representation_type': 'full_conversation', 'conversation_count': 26388}
Recovered embedding matrix from FAISS (no re-embedding): (26388, 768)


## Step 2 — Reproduce 004's candidate geometry (distances + ambiguity margins)

Same recipe as 004 (`normalized KMeans, k=10, n_init=20, seed=42`) applied to the
recovered vectors. Reproduced cluster ids are **aligned** to 004's candidate intents
by majority vote (robust to label permutation); candidate labels themselves always
come from 004's file. Per-conversation signals:

- `d_best`, `d_second` — euclidean distance to nearest / second-nearest centroid
- `ambiguity_margin = d_second - d_best` — **small = ambiguous** (a distance margin,
  never called a calibrated probability)


In [3]:
Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
km = KMeans(n_clusters=SELECTED_K, n_init=20, random_state=SEED)
geo = km.fit_predict(Xn)
dists = km.transform(Xn)  # euclidean distance to each centroid, same space KMeans optimizes
order = np.argsort(dists, axis=1)
d_best = dists[np.arange(len(Xn)), order[:, 0]]
d_second = dists[np.arange(len(Xn)), order[:, 1]]
margin = d_second - d_best
best_cluster = order[:, 0]
second_cluster = order[:, 1]

geo_sizes = pd.Series(geo).value_counts().sort_index()
print("Reproduced cluster sizes:", geo_sizes.tolist())
# 004's partition was [3233, 2224, 2907, 1728, 4388, 2039, 1958, 2453, 2553, 2905]
assert geo_sizes.tolist() == [3233, 2224, 2907, 1728, 4388, 2039, 1958, 2453, 2553, 2905], \
    "geometry does not reproduce 004's partition — stop before sampling"

cand_intents = np.array([r.get("intent") or "Unclassified" for r in records])
contingency = pd.crosstab(pd.Series(geo, name="geo_cluster"), pd.Series(cand_intents, name="candidate_intent"))
cluster_to_intent = {c: contingency.loc[c].idxmax() for c in contingency.index}
purity = {c: contingency.loc[c].max() / contingency.loc[c].sum() for c in contingency.index}
print("\nGeo-cluster -> candidate-intent alignment (majority vote):")
display(pd.DataFrame({"candidate_intent": pd.Series(cluster_to_intent),
                      "purity": pd.Series(purity).round(4),
                      "size": geo_sizes}))
assert min(purity.values()) > 0.90, "alignment ambiguous — inspect contingency before proceeding"

cand_counts = pd.Series(cand_intents).value_counts()
cand_dist = pd.DataFrame({"count_in_corpus": cand_counts,
                          "percentage_in_corpus": (cand_counts / len(records) * 100).round(2)})
print("\nCURRENT CANDIDATE DISTRIBUTION (PROVISIONAL — unsupervised, not ground truth):")
display(cand_dist)


Reproduced cluster sizes: [3233, 2224, 2907, 1728, 4388, 2039, 1958, 2453, 2553, 2905]

Geo-cluster -> candidate-intent alignment (majority vote):


,candidate_intent,purity,size
0,Positive experience,1.0,3233
1,Social acknowledgement,1.0,2224
2,Disruption recovery,1.0,2907
3,Positive experience,1.0,1728
4,Flight delay,1.0,4388
5,Seats and fares,1.0,2039
6,Baggage issues,1.0,1958
7,Service complaint,1.0,2453
8,General dissatisfaction,1.0,2553
9,Flight information,1.0,2905



CURRENT CANDIDATE DISTRIBUTION (PROVISIONAL — unsupervised, not ground truth):


,count_in_corpus,percentage_in_corpus
Positive experience,4961,18.80
Flight delay,4388,16.63
Disruption recovery,2907,11.02
Flight information,2905,11.01
General dissatisfaction,2553,9.67
Service complaint,2453,9.30
Social acknowledgement,2224,8.43
Seats and fares,2039,7.73
Baggage issues,1958,7.42


## Step 3 — Representative sample (150, provisional-distribution quotas)

Quotas per candidate intent follow the **PROVISIONAL** corpus distribution via the
**largest-remainder method** (exact total 150; no forced equal-per-intent counts).
Within each intent, selection is **diversity-aware, not centroid-greedy**: members are
sorted by distance-to-centroid, spread across inner / mid / outer bands
(50% / 30% / 20%), and taken at evenly spaced positions, so the 150 are *reasonably
typical* without being near-duplicates. Exact-duplicate texts are removed globally first.


In [4]:
# ---- global exact-duplicate guard (normalized whitespace/case) ----
def norm_text_key(record):
    return re.sub(r"\s+", " ", format_conversation_readable(record).lower()).strip()

seen, dup_count, keep_mask = set(), 0, np.ones(len(records), dtype=bool)
for i, r in enumerate(records):
    key = norm_text_key(r)
    if key in seen:
        keep_mask[i] = False
        dup_count += 1
    else:
        seen.add(key)
print(f"Exact-duplicate conversations excluded from sampling pool: {dup_count:,}")
eligible = np.where(keep_mask)[0]

# ---- largest-remainder quotas for exactly 150 ----
intents_sorted = cand_counts.index.tolist()  # corpus-frequency order
raw_quota = {it: N_REPRESENTATIVE * int(cand_counts[it]) / len(records) for it in intents_sorted}
quota = {it: int(np.floor(raw_quota[it])) for it in intents_sorted}
remainder = N_REPRESENTATIVE - sum(quota.values())
for it in sorted(intents_sorted, key=lambda t: raw_quota[t] - quota[t], reverse=True)[:remainder]:
    quota[it] += 1
assert sum(quota.values()) == N_REPRESENTATIVE
print("Largest-remainder quotas (PROVISIONAL distribution -> 150):")
display(pd.DataFrame({"corpus_pct": (cand_counts / len(records) * 100).round(2),
                      "target_count": pd.Series(quota)}))

# ---- diversity-aware pick within one intent ----
def cosine_sim_matrix(A, B):
    return A @ B.T  # inputs are L2-normalized

def diverse_pick_within_intent(pool, k, centroid_vec, label):
    idx = np.asarray(pool)
    d = np.linalg.norm(Xn[idx] - centroid_vec, axis=1)
    srt = idx[np.argsort(d, kind="stable")]
    n = len(srt)
    bands = [(srt[:max(1, int(0.5 * n))], max(0, int(round(0.50 * k)))),
             (srt[max(1, int(0.5 * n)):max(1, int(0.8 * n))], max(0, int(round(0.30 * k)))),
             (srt[max(1, int(0.8 * n)):], 0)]
    bands[2] = (bands[2][0], max(0, k - bands[0][1] - bands[1][1]))
    picked, picked_vecs = [], []
    for band, need in bands:
        if need <= 0 or len(band) == 0:
            continue
        pos = np.linspace(0, len(band) - 1, min(need, len(band))).round().astype(int)
        for p in band[pos]:
            if len(picked) >= k:
                break
            v = Xn[p]
            if picked_vecs and (cosine_sim_matrix(v[None, :], np.stack(picked_vecs)).max() > 0.97):
                continue  # near-duplicate guard
            picked.append(int(p))
            picked_vecs.append(v)
    # top-up (extremely rare) without guard if banding under-filled
    if len(picked) < k:
        for p in srt:
            if int(p) not in picked:
                picked.append(int(p))
            if len(picked) >= k:
                break
    assert len(picked) == k, (label, len(picked), k)
    return picked

centroids = km.cluster_centers_ / np.maximum(np.linalg.norm(km.cluster_centers_, axis=1, keepdims=True), 1e-12)
rep_idx = []
for it in intents_sorted:
    pool = [i for i in eligible if cand_intents[i] == it]
    assert len(pool) >= quota[it], (it, len(pool), quota[it])
    cl = contingency[it].idxmax()  # geo cluster that maps to this candidate intent
    rep_idx += diverse_pick_within_intent(pool, quota[it], centroids[cl], it)
rep_idx = np.array(sorted(rep_idx))
assert len(set(rep_idx)) == N_REPRESENTATIVE
print(f"Representative selected: {len(rep_idx)} (quota check: {Counter(cand_intents[i] for i in rep_idx)})")


Exact-duplicate conversations excluded from sampling pool: 0
Largest-remainder quotas (PROVISIONAL distribution -> 150):


,corpus_pct,target_count
Positive experience,18.80,28
Flight delay,16.63,25
Disruption recovery,11.02,17
Flight information,11.01,16
General dissatisfaction,9.67,14
Service complaint,9.30,14
Social acknowledgement,8.43,13
Seats and fares,7.73,12
Baggage issues,7.42,11


Representative selected: 150 (quota check: Counter({np.str_('Positive experience'): 28, np.str_('Flight delay'): 25, np.str_('Disruption recovery'): 17, np.str_('Flight information'): 16, np.str_('General dissatisfaction'): 14, np.str_('Service complaint'): 14, np.str_('Social acknowledgement'): 13, np.str_('Seats and fares'): 12, np.str_('Baggage issues'): 11}))


## Step 4 — Uncertainty / diversity sample (60, ambiguity-margin driven)

Ranked by ascending `ambiguity_margin` (small margin = close to two centroids =
ambiguous). Greedy diversity guard (cosine < 0.90 to already-chosen, relaxed only if
needed) plus a cap of 8 per (best, second-best) cluster pair keeps the 60 from
collapsing into one confusing region. Reported as an **uncertainty / ambiguity score**,
never a calibrated probability.


In [5]:
taken = set(int(i) for i in rep_idx)
unc_pool = np.array([i for i in eligible if int(i) not in taken])
unc_pool = unc_pool[np.argsort(margin[unc_pool], kind="stable")]  # most ambiguous first

unc_idx, pair_counts = [], Counter()
for acomp in [0.90, 0.95, 2.0]:  # progressively relax diversity guard if quota unmet
    for i in unc_pool:
        if len(unc_idx) >= N_UNCERTAINTY:
            break
        i = int(i)
        if i in taken or i in unc_idx:
            continue
        pair = (int(best_cluster[i]), int(second_cluster[i]))
        if pair_counts[pair] >= 8:
            continue
        if unc_idx and acomp < 2.0:
            sims = cosine_sim_matrix(Xn[i][None, :], Xn[np.array(unc_idx)])
            if sims.max() >= acomp:
                continue
        unc_idx.append(i)
        pair_counts[pair] += 1
    if len(unc_idx) >= N_UNCERTAINTY:
        break
assert len(unc_idx) == N_UNCERTAINTY, len(unc_idx)
taken.update(unc_idx)
print(f"Uncertainty selected: {len(unc_idx)}")
print(f"margin range in selection: [{margin[np.array(unc_idx)].min():.4f}, {margin[np.array(unc_idx)].max():.4f}] "
      f"vs corpus median {np.median(margin):.4f} (selection << population => genuinely ambiguous)")
print("Distinct best-clusters covered:", len({int(best_cluster[i]) for i in unc_idx}))
display(pd.Series([cand_intents[i] for i in unc_idx]).value_counts().to_frame("uncertainty_count"))


Uncertainty selected: 60
margin range in selection: [0.0000, 0.0003] vs corpus median 0.0282 (selection << population => genuinely ambiguous)
Distinct best-clusters covered: 10


,uncertainty_count
Flight information,16
Disruption recovery,11
Positive experience,8
Seats and fares,6
General dissatisfaction,5
Flight delay,4
Social acknowledgement,4
Service complaint,3
Baggage issues,3


## Step 5 — Hard / challenge sample (40, difficulty-heuristic driven)

Deliberately stress-tests the taxonomy with a transparent **heuristic difficulty score**
(weights sum to 1; heuristics, not truth):

| Component | Weight | Captures |
|---|---|---|
| shortness (`1 - min(1, chars/300)`) | 0.20 | very short conversations |
| few messages (`1 - min(1, n_msg/4)`) | 0.10 | low-context exchanges |
| vagueness (generic-phrase hits) | 0.15 | vague / generic wording |
| multi-intent (distinct keyword-group hits) | 0.20 | plausibly fits >1 intent |
| sarcasm / indirect cues | 0.10 | sarcasm, indirect complaints |
| outlierness (within-cluster z of `d_best`) | 0.10 | boundary / outlier examples |
| low-quality cluster (`General dissatisfaction`, `Social acknowledgement`) | 0.10 | likely-ambiguous current labels |
| overlap (`margin` < corpus median) | 0.05 | high-overlap intents |

Same cosine diversity guard (< 0.90) as the uncertainty set.


In [6]:
VAGUE_PATTERNS = [r"\bplease help\b", r"\bdm\b", r"\bthanks?\b$", r"\bok\b",
                  r"\bwhy\b.*\?", r"\bwhat\b.*\?", r"^hi\b", r"^hello\b",
                  r"\bneed help\b", r"\bcan you help\b"]
MULTI_GROUPS = {
    "delay": ["delay", "late", "waiting", "tarmac", "maintenance", "gate"],
    "cancel": ["cancel", "rebook", "missed connection", "compensation", "refund"],
    "baggage": ["baggage", "luggage", "\bbag\b", "lost", "damaged"],
    "seat": ["seat", "upgrade", "fare", "basic economy", "legroom"],
    "service": ["agent", "staff", "rude", "customer service", "complaint"],
    "info": ["status", "policy", "schedule", "terminal", "check-in", "check in"],
    "praise": ["thank", "great", "amazing", "kudos", "\blove\b"],
}
SARCASM_CUES = ["thanks a lot", "great job", "good job", "nice job", "wow", "lol",
                "sure", "love how", "thanks for nothing", "awesome.*not"]
LOWQ_INTENTS = {"General dissatisfaction", "Social acknowledgement"}
MED_MARGIN = float(np.median(margin))

def difficulty_features(i):
    r = records[i]
    cust = " ".join(str(m.get("text", "")) for m in r["messages"] if m.get("role") == "customer").lower()
    full = format_conversation_readable(r)
    n_msg = len(r["messages"])
    chars = len(full)
    vague = sum(1 for p in VAGUE_PATTERNS if re.search(p, cust)) / 2.0
    multi = sum(1 for kws in MULTI_GROUPS.values() if any(re.search(k, cust) for k in kws)) / 3.0
    sarc = sum(1 for c in SARCASM_CUES if c in cust) / 1.0
    cl = int(best_cluster[i])
    members = margin[geo == cl]
    outlier = float(np.clip((margin[i] - members.mean()) / (members.std() + 1e-9) * -1.0, 0, 1))
    # note: NEGATED z on margin -> far-from-centroid members score high; recompute on d_best:
    dm = d_best[geo == cl]
    outlier = float(np.clip((d_best[i] - dm.mean()) / (dm.std() + 1e-9), 0, 3) / 3.0)
    lowq = 1.0 if cand_intents[i] in LOWQ_INTENTS else 0.0
    overlap = 1.0 if margin[i] < MED_MARGIN else 0.0
    short = 1.0 - min(1.0, chars / 300.0)
    few = 1.0 - min(1.0, n_msg / 4.0)
    score = (0.20 * short + 0.10 * few + 0.15 * min(1.0, vague) + 0.20 * min(1.0, multi)
             + 0.10 * min(1.0, sarc) + 0.10 * outlier + 0.10 * lowq + 0.05 * overlap)
    return {"short": short, "few": few, "vague": min(1.0, vague), "multi": min(1.0, multi),
            "sarcasm": min(1.0, sarc), "outlier": outlier, "lowq": lowq,
            "overlap": overlap, "score": score}

chal_pool = np.array([i for i in eligible if int(i) not in taken])
chal_feat = {int(i): difficulty_features(int(i)) for i in chal_pool}
chal_pool = chal_pool[np.argsort([-chal_feat[int(i)]["score"] for i in chal_pool], kind="stable")]

chal_idx = []
for acomp in [0.90, 0.95, 2.0]:
    for i in chal_pool:
        if len(chal_idx) >= N_CHALLENGE:
            break
        i = int(i)
        if i in taken or i in chal_idx:
            continue
        if chal_idx and acomp < 2.0:
            if cosine_sim_matrix(Xn[i][None, :], Xn[np.array(chal_idx)]).max() >= acomp:
                continue
        chal_idx.append(i)
    if len(chal_idx) >= N_CHALLENGE:
        break
assert len(chal_idx) == N_CHALLENGE, len(chal_idx)
taken.update(chal_idx)
sel_scores = [chal_feat[i]["score"] for i in chal_idx]
print(f"Challenge selected: {len(chal_idx)} | difficulty score "
      f"mean {np.mean(sel_scores):.3f} vs corpus-sample mean "
      f"{np.mean([difficulty_features(int(i))['score'] for i in rng.choice(chal_pool, 500, replace=False)]):.3f} "
      f"(selection >> random => genuinely difficult)")
display(pd.Series([cand_intents[i] for i in chal_idx]).value_counts().to_frame("challenge_count"))


Challenge selected: 40 | difficulty score mean 0.551 vs corpus-sample mean 0.198 (selection >> random => genuinely difficult)


,challenge_count
General dissatisfaction,29
Disruption recovery,4
Social acknowledgement,4
Baggage issues,1
Positive experience,1
Flight information,1


## Step 6 — Assemble one golden set (250) + distribution analysis

Single `golden_set.csv` with full sampling metadata. Columns keep the
**candidate suggestion** (`candidate_*`) strictly separate from the
**human gold label** (`human_*`, empty until reviewed). Checks: exactly 250 unique
conversations, disjoint groups, no exact-duplicate texts, all candidate labels preserved.


In [7]:
gold_rows = []
def add_rows(idxs, group):
    for i in sorted(int(x) for x in idxs):
        r = records[i]
        gold_rows.append({
            "conversation_id": str(r["conversation_id"]),
            "sampling_group": group,
            "candidate_cluster": int(best_cluster[i]),
            "candidate_second_cluster": int(second_cluster[i]),
            "candidate_intent": str(cand_intents[i]),
            "candidate_domain": str(r.get("domain") or ""),
            "candidate_sub_intent": str(r.get("sub_intent") or ""),
            "candidate_confidence": str(r.get("intent_confidence") or ""),
            "candidate_distance_best": round(float(d_best[i]), 4),
            "candidate_ambiguity_margin": round(float(margin[i]), 4),
            "difficulty_score": round(float(difficulty_features(int(i))["score"]), 4),  # heuristic, all rows (review sorting)
            "n_messages": len(r["messages"]),
            "full_conversation_text": format_conversation_readable(r),
            "human_domain": "",
            "human_intent": "",
            "human_sub_intent": "",
            "annotation_status": "pending",
            "annotation_outcome": "",
            "reviewer_notes": "",
            "human_intent_r2": "",
            "second_pass": False,
        })

add_rows(rep_idx, "representative")
add_rows(unc_idx, "uncertainty")
add_rows(chal_idx, "challenge")
gold = pd.DataFrame(gold_rows)

# deterministic 50-record second-pass subset (30 / 12 / 8), stratified by group
for grp, k in [("representative", 30), ("uncertainty", 12), ("challenge", 8)]:
    ids = gold.index[gold["sampling_group"] == grp].to_numpy()
    gold.loc[rng.choice(ids, k, replace=False), "second_pass"] = True
assert gold["second_pass"].sum() == 50

# ---- quality gates before save ----
assert len(gold) == GOLD_SET_SIZE, len(gold)
assert gold["conversation_id"].nunique() == GOLD_SET_SIZE, "duplicate conversation_id"
assert gold["full_conversation_text"].map(lambda t: re.sub(r"\s+", " ", t.lower()).strip()).nunique() == GOLD_SET_SIZE, \
    "exact-duplicate texts in golden set"
assert set(gold["sampling_group"]) == {"representative", "uncertainty", "challenge"}
assert Counter(gold["sampling_group"]) == {"representative": 150, "uncertainty": 60, "challenge": 40}
assert (gold["candidate_intent"] != "") .all() and (gold["candidate_intent"] != "Unclassified").all()
assert not set(gold["conversation_id"]) - set(rec_by_id), "sampling leakage: unknown ids"
for c in ["human_intent", "human_sub_intent", "human_domain", "annotation_status", "reviewer_notes"]:
    assert c in gold.columns, c

GOLDEN_PATH.parent.mkdir(parents=True, exist_ok=True)
gold.to_csv(GOLDEN_PATH, index=False)
print(f"Saved {GOLDEN_PATH} — {len(gold)} rows, {len(gold.columns)} columns")

print("\nTOTAL GOLD SET (sampling_distribution — a design, not a population estimate):")
display(gold["sampling_group"].value_counts().to_frame("count")
        .assign(percent=lambda f: (f["count"] / len(gold) * 100).round(1)))
print("\nREPRESENTATIVE vs PROVISIONAL corpus distribution:")
rep = gold[gold["sampling_group"] == "representative"]["candidate_intent"].value_counts()
display(pd.DataFrame({"corpus_count": cand_counts, "corpus_pct": (cand_counts / len(records) * 100).round(2),
                      "rep_selected": rep, "rep_pct": (rep / N_REPRESENTATIVE * 100).round(2)}).fillna(0))
print("\nUNCERTAINTY distribution (intentionally enriched):")
display(gold[gold["sampling_group"] == "uncertainty"]["candidate_intent"].value_counts().to_frame("count"))
print("\nCHALLENGE distribution (intentionally enriched):")
display(gold[gold["sampling_group"] == "challenge"]["candidate_intent"].value_counts().to_frame("count"))


Saved ..\data\processed\golden_set.csv — 250 rows, 21 columns

TOTAL GOLD SET (sampling_distribution — a design, not a population estimate):


,count,percent
sampling_group,,
representative,150,60.0
uncertainty,60,24.0
challenge,40,16.0



REPRESENTATIVE vs PROVISIONAL corpus distribution:


,corpus_count,corpus_pct,rep_selected,rep_pct
Positive experience,4961,18.80,28,18.67
Flight delay,4388,16.63,25,16.67
Disruption recovery,2907,11.02,17,11.33
Flight information,2905,11.01,16,10.67
General dissatisfaction,2553,9.67,14,9.33
Service complaint,2453,9.30,14,9.33
Social acknowledgement,2224,8.43,13,8.67
Seats and fares,2039,7.73,12,8.00
Baggage issues,1958,7.42,11,7.33



UNCERTAINTY distribution (intentionally enriched):


,count
candidate_intent,
Flight information,16
Disruption recovery,11
Positive experience,8
Seats and fares,6
General dissatisfaction,5
Flight delay,4
Social acknowledgement,4
Baggage issues,3
Service complaint,3



CHALLENGE distribution (intentionally enriched):


,count
candidate_intent,
General dissatisfaction,29
Disruption recovery,4
Social acknowledgement,4
Flight information,1
Baggage issues,1
Positive experience,1


## Step 7 — Candidate-taxonomy review support (from 004, verbatim)

Review the proposal **before** labeling. For each candidate intent decide:
**KEEP / RENAME / MERGE / SPLIT / REMOVE (non-actionable)**. The per-intent cards below
show the 004 definition, corpus size, representative conversation ids, and the most
confusing neighbor intents (mined from the uncertainty set's top/second cluster pairs).


In [8]:
# ---- 004's candidate taxonomy, reproduced verbatim (do NOT edit here — review it) ----
CLUSTER_TO_TAXONOMY = {
    0: {"domain": "Experience", "intent": "Positive experience", "sub_intent": "Flight or crew praise"},
    1: {"domain": "Social", "intent": "Social acknowledgement", "sub_intent": "Thanks or casual mention"},
    2: {"domain": "Flight", "intent": "Disruption recovery", "sub_intent": "Cancellation, rebooking, missed connection, or compensation"},
    3: {"domain": "Experience", "intent": "Positive experience", "sub_intent": "Flight or crew praise"},
    4: {"domain": "Flight", "intent": "Flight delay", "sub_intent": "Late departure, arrival, gate, or tarmac wait"},
    5: {"domain": "Seating and fares", "intent": "Seats and fares", "sub_intent": "Seat assignment, upgrade, basic fare, or paid seating"},
    6: {"domain": "Baggage", "intent": "Baggage issues", "sub_intent": "Lost, delayed, damaged, checked, or carry-on baggage"},
    7: {"domain": "Customer service", "intent": "Service complaint", "sub_intent": "Agent, staff, responsiveness, or support quality"},
    8: {"domain": "Customer service", "intent": "General dissatisfaction", "sub_intent": "Broad negative experience without one dominant issue"},
    9: {"domain": "Travel information", "intent": "Flight information", "sub_intent": "Flight status, policy, route, schedule, or general question"},
}
INTENT_DEFINITIONS = {
    "Positive experience": "Praise or thanks for a completed flight, crew, or staff experience.",
    "Social acknowledgement": "Short social contact, thanks, casual mention, or low-context exchange without a support problem.",
    "Disruption recovery": "Cancellation, missed connection, rerouting, rebooking, compensation, or recovery after a disrupted itinerary.",
    "Flight delay": "Late departure, arrival, gate, maintenance wait, or tarmac delay is the dominant issue.",
    "Seats and fares": "Seat assignment/change, upgrade, cabin, basic-economy restriction, or paid seating/fare issue.",
    "Baggage issues": "Lost, delayed, damaged, checked, carry-on, or luggage-handling issue.",
    "Service complaint": "Complaint about agent behavior, staff, responsiveness, customer relations, or support quality.",
    "General dissatisfaction": "Broad negative experience where no single operational issue dominates; do not use when a specific issue applies.",
    "Flight information": "Question or request about flight status, route, schedule, policy, check-in, or travel rules.",
}
ANNOTATION_GUIDE = {
    "Positive experience": ("Include: explicit praise of crew/flight/service. Exclude: bare 'thanks' with no content (that is Social).",
                            "Confusing: Social acknowledgement."),
    "Social acknowledgement": ("Include: greetings, bare thanks, casual mentions, no problem stated. Exclude: any actionable request.",
                               "Confusing: Positive experience, Flight information."),
    "Disruption recovery": ("Include: cancelled/missed/rebooked itinerary + recovery ask. Exclude: mere delay with original flight intact (that is Flight delay).",
                            "Confusing: Flight delay, Flight information."),
    "Flight delay": ("Include: late departure/arrival/gate/tarmac wait as the dominant issue. Exclude: delay that already caused a missed connection (that is Disruption recovery).",
                     "Confusing: Disruption recovery, Service complaint."),
    "Seats and fares": ("Include: seat assignment/change, upgrade, cabin, fare rules/charges. Exclude: general delay complaints from a seat (that is Flight delay).",
                        "Confusing: Flight information, Service complaint. Watch for SPLIT into seat-assignment vs fare/payment."),
    "Baggage issues": ("Include: lost/delayed/damaged/checked/carry-on baggage handling. Exclude: complaining about staff while discussing bags -> pick the dominant issue.",
                       "Confusing: Service complaint."),
    "Service complaint": ("Include: complaint about agent/staff/responsiveness/support quality. Exclude: broad venting with no service target (that is General dissatisfaction).",
                          "Confusing: General dissatisfaction."),
    "General dissatisfaction": ("Include ONLY: broad negativity where no single operational issue dominates. Exclude: any case a specific intent fits — prefer the specific intent.",
                                "Confusing: Service complaint. Top MERGE/REMOVE suspect."),
    "Flight information": ("Include: questions about status/route/schedule/policy/check-in. Exclude: complaints phrased as questions about a past failure.",
                           "Confusing: Disruption recovery, Seats and fares."),
}

# confusing-neighbor mining from the uncertainty set's (best, second-best) intent pairs
pair_counter = Counter()
for i in unc_idx:
    a, b = cluster_to_intent[int(best_cluster[int(i)])], cluster_to_intent[int(second_cluster[int(i)])]
    pair_counter[tuple(sorted((a, b)))] += 1
print("Top confusing candidate-intent pairs (from 60 uncertainty cases):")
for (a, b), c in pair_counter.most_common(8):
    print(f"  {a}  <->  {b}: {c}")
neighbors = defaultdict(set)
for (a, b) in pair_counter:
    neighbors[a].add(b); neighbors[b].add(a)

rep_examples = gold[gold["sampling_group"] == "representative"].groupby("candidate_intent")["conversation_id"].apply(list)
print("\n================ CANDIDATE TAXONOMY CARDS (004 proposal — KEEP / RENAME / MERGE / SPLIT / REMOVE) ================")
for intent in cand_counts.index:
    dom = next(v["domain"] for v in CLUSTER_TO_TAXONOMY.values() if v["intent"] == intent)
    clus = sorted(c for c, v in CLUSTER_TO_TAXONOMY.items() if v["intent"] == intent)
    ex = (rep_examples.get(intent, [])[:3])
    inc, _ = ANNOTATION_GUIDE[intent]
    print(f"\n### {intent}  [domain={dom} | clusters={clus} | corpus={int(cand_counts[intent]):,} "
          f"({cand_counts[intent] / len(records) * 100:.1f}%)]")
    print(f"Definition: {INTENT_DEFINITIONS[intent]}")
    print(f"Guidance: {inc}")
    print(f"Known confusing neighbors: {sorted(neighbors.get(intent, [])) or 'none observed'}")
    print(f"Representative golden-set ids: {ex}")
print("\nIf several reviewed examples need an intent that is not listed, use human label "
      "'New intent (see notes)' — repeated NEW INTENT is evidence the taxonomy must expand.")


Top confusing candidate-intent pairs (from 60 uncertainty cases):
  Disruption recovery  <->  Flight delay: 8
  Disruption recovery  <->  Flight information: 7
  Positive experience  <->  Social acknowledgement: 7
  Disruption recovery  <->  General dissatisfaction: 6
  Positive experience  <->  Positive experience: 4
  Flight information  <->  Positive experience: 3
  Flight information  <->  Service complaint: 3
  Flight information  <->  General dissatisfaction: 3

================ CANDIDATE TAXONOMY CARDS (004 proposal — KEEP / RENAME / MERGE / SPLIT / REMOVE) ================

### Positive experience  [domain=Experience | clusters=[0, 3] | corpus=4,961 (18.8%)]
Definition: Praise or thanks for a completed flight, crew, or staff experience.
Guidance: Include: explicit praise of crew/flight/service. Exclude: bare 'thanks' with no content (that is Social).
Known confusing neighbors: ['Flight information', 'Positive experience', 'Social acknowledgement']
Representative golden-set ids:

## Step 8 — Human review interface

**You are reviewing the candidate label, never forced to accept it.** Per-row outcomes:
`ACCEPT` · `RENAME` · `CHANGE_INTENT` · `SPLIT_NEEDED` · `MERGE_NEEDED` · `SOCIAL` ·
`NON_ACTIONABLE` · `UNKNOWN` (unknown / insufficient context), plus free
`human_domain` / `human_intent` / `human_sub_intent` / `reviewer_notes`.

Two equivalent workflows (no widget dependency, no source-code editing per conversation):

- **A. In-notebook:** `get_review_view(...)` to find records (views A–H below),
  `show_conversation(id)` to read the full thread, `set_review(...)` to label — saves to CSV.
- **B. Spreadsheet:** open `data/processed/golden_set.csv`, edit only the `human_*`,
  `annotation_status/outcome`, `reviewer_notes` (+ optional `human_intent_r2`) columns,
  then run the reload cell. `load_reviews()` validates every edit.

Available views: **A** all · **B** representative · **C** uncertainty · **D** challenge ·
**E** by candidate intent · **F** by candidate cluster · **G** lowest-confidence first ·
**H** highest ambiguity first.


In [9]:
VALID_OUTCOMES = ["ACCEPT", "RENAME", "CHANGE_INTENT", "SPLIT_NEEDED",
                 "MERGE_NEEDED", "SOCIAL", "NON_ACTIONABLE", "UNKNOWN"]
CANDIDATE_INTENTS = cand_counts.index.tolist()
HUMAN_INTENTS_ALLOWED = CANDIDATE_INTENTS + ["Non-actionable",
    "Unknown / insufficient context", "New intent (see notes)"]

def load_reviews():
    global gold
    g = pd.read_csv(GOLDEN_PATH, dtype={"conversation_id": str}, keep_default_na=False)
    bad_out = set(g["annotation_outcome"]) - set(VALID_OUTCOMES + [""])
    assert not bad_out, f"invalid annotation_outcome values: {bad_out}"
    bad_h = set(g["human_intent"]) - set(HUMAN_INTENTS_ALLOWED + [""])
    assert not bad_h, f"invalid human_intent values (use 'New intent (see notes)' + notes): {bad_h}"
    rev = g[g["annotation_status"] == "reviewed"]
    unlabeled = rev[(rev["human_intent"] == "") &
                    (~rev["annotation_outcome"].isin(["UNKNOWN", "NON_ACTIONABLE", "SOCIAL", "SPLIT_NEEDED", "MERGE_NEEDED"]))]
    assert len(unlabeled) == 0, f"{len(unlabeled)} reviewed rows lack a human label/outcome"
    gold = g
    print(f"Loaded {len(g)} rows ({(g['annotation_status'] == 'reviewed').sum()} reviewed). Edits validated.")
    return g

def get_review_view(group=None, intent=None, cluster=None, sort_by="ambiguity", n=None, preview_chars=160):
    v = gold.copy()
    if group is not None:
        v = v[v["sampling_group"] == group]
    if intent is not None:
        v = v[v["candidate_intent"] == intent]
    if cluster is not None:
        v = v[v["candidate_cluster"] == int(cluster)]
    if sort_by == "ambiguity":      # H: highest ambiguity first (smallest margin)
        v = v.sort_values("candidate_ambiguity_margin")
    elif sort_by == "confidence":   # G: lowest confidence first (medium, then largest distance)
        v = v.sort_values(["candidate_confidence", "candidate_distance_best"], ascending=[True, False])
    elif sort_by == "difficulty":
        v = v.sort_values("difficulty_score", ascending=False)
    elif sort_by == "id":
        v = v.sort_values("conversation_id")
    else:
        raise ValueError("sort_by must be ambiguity | confidence | difficulty | id")
    if n is not None:
        v = v.head(n)
    out = v[["conversation_id", "sampling_group", "candidate_cluster", "candidate_intent",
             "candidate_confidence", "candidate_ambiguity_margin", "annotation_status",
             "annotation_outcome", "human_intent", "reviewer_notes"]].copy()
    out["conversation_preview"] = v["full_conversation_text"].str.replace(r"\s+", " ", regex=True).str[:preview_chars]
    return out

def show_conversation(conversation_id):
    row = gold[gold["conversation_id"] == str(conversation_id)]
    assert len(row) == 1, f"unknown conversation_id: {conversation_id}"
    r = row.iloc[0]
    print("=" * 78)
    print(f"ID {r['conversation_id']} | group={r['sampling_group']} | "
          f"candidate=[{r['candidate_cluster']}] {r['candidate_intent']} "
          f"(conf={r['candidate_confidence']}, margin={r['candidate_ambiguity_margin']}) "
          f"| second-best cluster={r['candidate_second_cluster']} | second_pass={r['second_pass']}")
    print(f"Current human label: domain={r['human_domain'] or '-'} intent={r['human_intent'] or '-'} "
          f"sub={r['human_sub_intent'] or '-'} | status={r['annotation_status']} "
          f"outcome={r['annotation_outcome'] or '-'} | notes={r['reviewer_notes'] or '-'}")
    print("-" * 78)
    print(r["full_conversation_text"])
    print("=" * 78)

def set_review(conversation_id, human_intent=None, human_domain=None, human_sub_intent=None,
               outcome=None, notes=None, reviewer2=False):
    assert outcome is None or outcome in VALID_OUTCOMES, f"invalid outcome: {outcome}"
    if human_intent is not None:
        assert human_intent in HUMAN_INTENTS_ALLOWED, f"invalid human_intent: {human_intent}"
    m = gold["conversation_id"] == str(conversation_id)
    assert m.sum() == 1, f"unknown conversation_id: {conversation_id}"
    col = "human_intent_r2" if reviewer2 else "human_intent"
    if human_intent is not None:
        gold.loc[m, col] = human_intent
    if human_domain is not None and not reviewer2:
        gold.loc[m, "human_domain"] = human_domain
    if human_sub_intent is not None and not reviewer2:
        gold.loc[m, "human_sub_intent"] = human_sub_intent
    if outcome is not None and not reviewer2:
        gold.loc[m, "annotation_outcome"] = outcome
        gold.loc[m, "annotation_status"] = "reviewed"
    if notes is not None and not reviewer2:
        gold.loc[m, "reviewer_notes"] = notes
    gold.to_csv(GOLDEN_PATH, index=False)
    n_rev = int((gold["annotation_status"] == "reviewed").sum())
    print(f"Saved {conversation_id} -> {GOLDEN_PATH} ({n_rev}/250 reviewed)")

def agreement_report():
    both = gold[(gold["human_intent"] != "") & (gold["human_intent_r2"] != "")]
    if len(both) == 0:
        print("No double-reviewed records yet: fill human_intent_r2 (e.g. via set_review(..., reviewer2=True)) "
              "on the 50 second_pass rows, then rerun. No agreement is claimed for single-reviewer data.")
        return None
    raw = float((both["human_intent"] == both["human_intent_r2"]).mean())
    out = {"n_double_reviewed": len(both), "raw_agreement": round(raw, 4), "cohen_kappa": None}
    if len(both) >= 2 and both["human_intent"].nunique() > 1:
        out["cohen_kappa"] = round(float(cohen_kappa_score(both["human_intent"], both["human_intent_r2"])), 4)
    print(f"Second-pass agreement: n={out['n_double_reviewed']}, raw={out['raw_agreement']}, kappa={out['cohen_kappa']}")
    display(pd.crosstab(both["human_intent"], both["human_intent_r2"]))
    return out

print("Review helpers ready: get_review_view / show_conversation / set_review / load_reviews / agreement_report")
print("Second-pass pool (50, stratified 30/12/8):",
      gold[gold['second_pass']].groupby('sampling_group')['conversation_id'].count().to_dict())
print("\nView H — 5 most ambiguous records:")
display(get_review_view(sort_by="ambiguity", n=5))


Review helpers ready: get_review_view / show_conversation / set_review / load_reviews / agreement_report
Second-pass pool (50, stratified 30/12/8): {'challenge': 8, 'representative': 30, 'uncertainty': 12}

View H — 5 most ambiguous records:


,conversation_id,sampling_group,candidate_cluster,candidate_intent,candidate_confidence,candidate_ambiguity_margin,annotation_status,annotation_outcome,human_intent,reviewer_notes,conversation_preview
175,2252870,uncertainty,9,Flight information,high,0.0,pending,,,,CUSTOMER: I'm not sure this plane is going into the right direction! @116125 @AmericanAir https://t.co/p49LiN36Bh AG...
174,2233561,uncertainty,4,Flight delay,high,0.0,pending,,,,CUSTOMER: Nothing worse than pushing back from the gate &amp; sitting for 40 minutes!!! Thanks @americanair #fail AG...
202,533606,uncertainty,0,Positive experience,high,0.0,pending,,,,CUSTOMER: Thanks @AmericanAir for the #upgrade to first class. DFW-LAX #losangeles #california #onlyflyAA #execplati...
188,2745104,uncertainty,7,Service complaint,high,0.0,pending,,,,CUSTOMER: @AmericanAir shocking service. AA ways to deal with complaints hang up on them 😡😡 AGENT: @768796 We're sor...
191,2829226,uncertainty,2,Disruption recovery,high,0.0,pending,,,,CUSTOMER: @AmericanAir and to make matters worse the next flight isn’t for 12 hours which means I’m missing a busine...


## Step 9 — Post-review taxonomy analysis (candidate → human)

Rerun this section after reviewing: it compares what 004 proposed against what the
human decided — a confusion-style diagnostic of the **taxonomy**, not classifier
accuracy. With zero reviews completed it reports status only.


In [19]:
reviewed = gold[gold["annotation_status"] == "reviewed"].copy()
print(f"Review completion: {len(reviewed)}/250 ({len(reviewed) / 250 * 100:.1f}%)")
if len(reviewed) == 0:
    print("No human labels yet — complete Step 8 reviews, then rerun from load_reviews().")
else:
    labeled = reviewed[reviewed["human_intent"] != ""].copy()
    print(f"\nHUMAN-ANNOTATED distribution (observed within reviewed, n={len(labeled)}):")
    display(labeled["human_intent"].value_counts().to_frame("human_count")
            .assign(human_pct=lambda f: (f["human_count"] / len(labeled) * 100).round(1)))

    print("\nCandidate -> human confusion (rows=candidate proposal, columns=human decision):")
    conf = pd.crosstab(labeled["candidate_intent"], labeled["human_intent"])
    display(conf)

    acc = (labeled["candidate_intent"] == labeled["human_intent"]).groupby(labeled["candidate_intent"]).mean()
    print("\nPer-candidate acceptance rate (human kept the proposed label):")
    display(acc.to_frame("acceptance_rate").round(3).sort_values("acceptance_rate"))

    print("\n--- Taxonomy revision flags ---")
    for intent, grp in labeled.groupby("candidate_intent"):
        n = len(grp)
        a = float((grp["candidate_intent"] == grp["human_intent"]).mean())
        spread = grp["human_intent"].value_counts(normalize=True)
        big = spread[spread >= 0.15]
        if n >= 5 and a < 0.50:
            print(f"MERGE/REDEFINE candidate: '{intent}' accepted only {a:.0%} (n={n})")
        if n >= 8 and len(big) >= 3:
            print(f"SPLIT candidate: '{intent}' spreads across {dict(big.round(2))} (n={n})")
    new_intent = labeled[labeled["human_intent"] == "New intent (see notes)"]
    if len(new_intent) >= 5:
        print(f"EXPANSION evidence: {len(new_intent)} NEW INTENT cases — inspect reviewer_notes for the missing concept(s)")
    elif len(new_intent) > 0:
        print(f"Note: {len(new_intent)} NEW INTENT case(s) so far")
    for outcome in ["SOCIAL", "NON_ACTIONABLE", "UNKNOWN"]:
        k = int((reviewed["annotation_outcome"] == outcome).sum())
        if k:
            print(f"{outcome}: {k} reviewed records")
    print(f"\nAccepted unchanged (ACCEPT): {int((reviewed['annotation_outcome'] == 'ACCEPT').sum())}")
    print(f"Corrected (RENAME/CHANGE_INTENT): "
          f"{int(reviewed['annotation_outcome'].isin(['RENAME', 'CHANGE_INTENT']).sum())}")


Review completion: 0/250 (0.0%)
No human labels yet — complete Step 8 reviews, then rerun from load_reviews().


## Step 10 — Quality checks + final report

Gates: exactly 250 unique conversations · required columns · no duplicate ids ·
no missing `sampling_group` · source data untouched (read-only use of 004 outputs) ·
candidate labels preserved · reviewed rows carry a human label or explicit outcome ·
no invalid intent names · no sampling leakage. Downstream rule: the 150
representative rows are the population-like evaluation subset; any single metric over
all 250 must be labeled an **enriched gold-set metric**, not population performance.


In [21]:
# ---- reload from disk so the report reflects the saved reviewer state ----
check = pd.read_csv(GOLDEN_PATH, dtype={"conversation_id": str}, keep_default_na=False)
REQUIRED = ["conversation_id", "full_conversation_text", "sampling_group", "candidate_cluster",
            "candidate_intent", "candidate_confidence", "candidate_ambiguity_margin",
            "human_intent", "human_sub_intent", "human_domain", "annotation_status", "reviewer_notes"]
missing = [c for c in REQUIRED if c not in check.columns]
assert not missing, f"missing columns: {missing}"
assert len(check) == 250 and check["conversation_id"].nunique() == 250
assert Counter(check["sampling_group"]) == {"representative": 150, "uncertainty": 60, "challenge": 40}
assert check["sampling_group"].isna().sum() == 0 and (check["sampling_group"] != "").all()
assert set(check["conversation_id"]) <= set(rec_by_id), "sampling leakage"
assert (check["candidate_intent"] == check["conversation_id"].map(
    lambda cid: rec_by_id[cid].get("intent"))).all(), "candidate labels were mutated"
rev = check[check["annotation_status"] == "reviewed"]
assert ((rev["human_intent"] != "") | (rev["annotation_outcome"] != "")).all()
assert set(check["human_intent"]) <= set(HUMAN_INTENTS_ALLOWED + [""])
assert set(check["annotation_outcome"]) <= set(VALID_OUTCOMES + [""])
print("ALL QUALITY CHECKS PASSED (250 unique · groups 150/60/40 · candidates preserved · labels valid · no leakage)")

n_rev = len(rev)
n_acc = int((rev["annotation_outcome"] == "ACCEPT").sum())
n_corr = int(rev["annotation_outcome"].isin(["RENAME", "CHANGE_INTENT"]).sum())
n_split = int((rev["annotation_outcome"] == "SPLIT_NEEDED").sum())
n_merge = int((rev["annotation_outcome"] == "MERGE_NEEDED").sum())
n_unk = int(rev["annotation_outcome"].isin(["UNKNOWN", "NON_ACTIONABLE", "SOCIAL"]).sum())
print("\n================ GOLDEN SET SUMMARY ================")
print(f"total = {len(check)} | representative = 150 | uncertainty/diversity = 60 | challenge = 40")
print(f"1. candidate taxonomy distribution: see Step 2 table (PROVISIONAL, n=26,388)")
print(f"2. golden-set sampling distribution: 150/60/40 by design (see Step 6 tables)")
print(f"3. human-reviewed distribution: {n_rev} reviewed; " +
      ("see Step 9 table" if n_rev else "no reviews yet — run Step 8"))
print(f"4. candidate -> human changes: see Step 9 confusion (needs reviews)")
print(f"5/6. merge / split candidates: see Step 9 flags (needs reviews)")
print(f"7. new intents discovered: {int((rev['human_intent'] == 'New intent (see notes)').sum())}")
print(f"8. ambiguous cases: 60 uncertainty (margin << median) + 40 challenge by design")
print(f"9. accepted unchanged: {n_acc} | 10. corrected: {n_corr}")
print(f"11. unknown/non-actionable/social: {n_unk} | split-needed: {n_split} | merge-needed: {n_merge}")
print(f"12. review completion: {n_rev}/250 ({n_rev / 250 * 100:.1f}%)")
print("\nDownstream: evaluate population-like behavior on the 150 representative rows; "
      "report any all-250 number as an ENRICHED gold-set metric.")
print(f"Deliverables: notebooks/005_golden_set.ipynb + {GOLDEN_PATH}")


ALL QUALITY CHECKS PASSED (250 unique · groups 150/60/40 · candidates preserved · labels valid · no leakage)

================ GOLDEN SET SUMMARY ================
total = 250 | representative = 150 | uncertainty/diversity = 60 | challenge = 40
1. candidate taxonomy distribution: see Step 2 table (PROVISIONAL, n=26,388)
2. golden-set sampling distribution: 150/60/40 by design (see Step 6 tables)
3. human-reviewed distribution: 0 reviewed; no reviews yet — run Step 8
4. candidate -> human changes: see Step 9 confusion (needs reviews)
5/6. merge / split candidates: see Step 9 flags (needs reviews)
7. new intents discovered: 0
8. ambiguous cases: 60 uncertainty (margin << median) + 40 challenge by design
9. accepted unchanged: 0 | 10. corrected: 0
11. unknown/non-actionable/social: 0 | split-needed: 0 | merge-needed: 0
12. review completion: 0/250 (0.0%)

Downstream: evaluate population-like behavior on the 150 representative rows; report any all-250 number as an ENRICHED gold-set metric.
